In [111]:
import pandas as pd


df_survey=pd.read_csv(r"..\Synthetic_Data\feedback_sentiment_analysis.csv")
df_survey["survey_date"] = pd.to_datetime(df_survey["survey_date"], format="%d-%m-%Y")
df_survey["performance_rating"] = df_survey["performance_rating"].astype(int)


#outliers

In [112]:
df_survey.columns

Index(['employee_id', 'survey_date', 'department', 'tenure_years',
       'performance_rating', 'engagement_score', 'feedback_comment',
       'feedback_category', 'sentiment', 'summary'],
      dtype='object')

In [113]:


numerical_cols=["tenure_years","performance_rating",]

cols = numerical_cols

outlier_counts = {}

for col in cols:
    Q1 = df_survey[col].quantile(0.25)
    Q3 = df_survey[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_survey[(df_survey[col] < lower_bound) | (df_survey[col] > upper_bound)]
    outlier_counts[col] = len(outliers)

# Print results
for col, count in outlier_counts.items():
    print(f"{col}: {count} outliers")
    



def cap_outliers_iqr(df, cols):
    df_copy = df.copy()
    
    for col in cols:
        Q1 = df_copy[col].quantile(0.25)
        Q3 = df_copy[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        df_copy[col] = df_copy[col].clip(lower, upper)
    
    return df_copy


df_survey=cap_outliers_iqr(df_survey, numerical_cols)


tenure_years: 0 outliers
performance_rating: 0 outliers


# skewness


In [114]:

print(df_survey[numerical_cols].skew())

## no column skewed


tenure_years          0.418142
performance_rating    0.036578
dtype: float64


### create target columns


In [115]:

df_survey = df_survey.sort_values(["employee_id", "survey_date"])

# target next years

target_years=[1]

for lag in target_years:
    df_survey[f"lag_{lag}_performance"] = df_survey.groupby("employee_id")["performance_rating"].shift(lag)
    df_survey[f"lag_{lag}_engagement"] = df_survey.groupby("employee_id")["engagement_score"].shift(lag)



In [116]:

def add_appearance_flag(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds a column 'appearance_next_survey' indicating whether an employee
    appears in the next survey date.

    1 -> Employee present in next survey
    0 -> Employee not present (possible attrition)

    Assumes:
    - 'employee_id' uniquely identifies employee
    - 'survey_date' is datetime
    """

    df = df.copy()

    # Ensure proper sorting
    df = df.sort_values(by=["employee_id", "survey_date"])

    # Shift survey_date per employee to get next record
    df["next_survey_date"] = df.groupby("employee_id")["survey_date"].shift(-1)

    # Flag: if next record exists → 1 else 0
    df["appearance_next_survey"] = df["next_survey_date"].notna().astype(int)

    # Drop helper column (clean output)
    df.drop(columns=["next_survey_date"], inplace=True)

    return df



df_survey=add_appearance_flag(df_survey)

## one hot encoding of numerical columns

In [117]:


import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

df = df_survey.copy()




cat_cols = ["department", "feedback_category", "sentiment"]
import pandas as pd
import category_encoders as ce


# create encoder
encoder = ce.OneHotEncoder(cols=cat_cols, use_cat_names=True)

# fit
encoder.fit(df)

# transform (returns DataFrame)
df = encoder.transform(df)



In [118]:
import joblib
joblib.dump(encoder, "onehot_encoder.pkl")

['onehot_encoder.pkl']

# performance model

In [119]:

x_columns=[  
    'department_HR', 'department_Operations','department_Finance', 'department_Sales', 'department_Marketing', 'department_Engineering', 
            'tenure_years', 'performance_rating', 'engagement_score',
       'feedback_category_Work-life balance','feedback_category_Career growth', 'feedback_category_Compensation','feedback_category_Culture', 'feedback_category_Management',
       'sentiment_Positive', 'sentiment_Negative', 'sentiment_Neutral',
       ]

        #  'lag_1_performance', 'lag_1_engagement'

y_column="lag_1_performance"

df2=df.copy()

df = df.dropna(subset=["lag_1_performance"])

X=df[x_columns]
y=df[y_column]

In [120]:

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


X_train.shape,X_test.shape


y_train = y_train.astype(int).astype(str)
y_test = y_test.astype(int).astype(str)

# check for correlation any drop any numerical column whose correlation is more that 0.85


In [121]:

def correlation_pairs(dataset, threshold):
    corr_matrix = dataset.corr()
    correlated_pairs = []

    # loop through matrix
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            corr_value = corr_matrix.iloc[i, j]
            if abs(corr_value) > threshold:
                col1 = corr_matrix.columns[i]
                col2 = corr_matrix.columns[j]
                correlated_pairs.append((col1, col2, corr_value))

    return correlated_pairs


## threshold--Domain expertise
corr_features=correlation_pairs(X_train[numerical_cols],0.85)

print(corr_features)

[]


In [122]:
from sklearn.preprocessing import StandardScaler
# scaler=StandardScaler()

# X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


# X_train_sc=X_train

# X_test_sc=X_test



In [123]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
from sklearn.pipeline import Pipeline


pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])



param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}




grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=3,
    refit=True
)


grid.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [8, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : 

In [124]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,f1_score,roc_auc_score


print("Best parameters:", grid.best_params_)


y_train_pred = grid.predict(X_train)
y_test_pred  = grid.predict(X_test)


print("--- Training Set ---")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("F1 Score:", f1_score(y_train, y_train_pred, average='weighted'))

# print("Classification Report:\n", classification_report(y_train, y_train_pred))
# print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))


print("--- Test Set ---")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1 Score:", f1_score(y_test, y_test_pred, average='weighted'))

# print("Classification Report:\n", classification_report(y_test, y_test_pred))
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))




Best parameters: {'model__max_depth': 15, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 100}
--- Training Set ---
Accuracy: 1.0
F1 Score: 1.0
--- Test Set ---
Accuracy: 0.7288135593220338
F1 Score: 0.7188275514976765


In [125]:

import joblib

joblib.dump(grid, "performance_model.pkl")

['performance_model.pkl']

# engagement model

In [126]:

x_columns=[  
    'department_HR', 'department_Operations','department_Finance', 'department_Sales', 'department_Marketing', 'department_Engineering', 
            'tenure_years', 'performance_rating', 'engagement_score',
       'feedback_category_Work-life balance','feedback_category_Career growth', 'feedback_category_Compensation','feedback_category_Culture', 'feedback_category_Management',
       'sentiment_Positive', 'sentiment_Negative', 'sentiment_Neutral',
       ]

        #  'lag_1_performance', 'lag_1_engagement'

y_column="lag_1_engagement"

df=df2.copy()

df = df.dropna(subset=["lag_1_engagement"])

X=df[x_columns]
y=df[y_column]

In [127]:

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


X_train.shape,X_test.shape




((232, 17), (59, 17))

In [128]:
from sklearn.preprocessing import StandardScaler
# scaler=StandardScaler()

# X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


# X_train_sc=X_train

# X_test_sc=X_test



In [129]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd


# rf = RandomForestRegressor(random_state=42)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(random_state=42))
])


# param_grid = {
#     'n_estimators': [200, 400, 300],   
#     'max_depth': [10, 20, None],               
#     'min_samples_split': [2, 5, 10],           
#     'min_samples_leaf': [1, 2, 4],              
#     'max_features': ['sqrt', 'log2'],      
# }


param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}




# 3️⃣ Grid Search
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=3,
    refit=True
)

# 4️⃣ Fit using scaled training data
grid.fit(X_train, y_train)



Fitting 5 folds for each of 108 candidates, totalling 540 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [8, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messag

In [130]:

# 5️⃣ Best parameters
print("Best parameters:", grid.best_params_)

# 6️⃣ Predictions (on scaled data)
y_train_pred = grid.predict(X_train)
y_test_pred = grid.predict(X_test)

# 7️⃣ Train metrics
mae_train = mean_absolute_error(y_train, y_train_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
rmse_train = np.sqrt(mse_train)
r2_train = r2_score(y_train, y_train_pred)

print("\n=== TRAIN METRICS ===")
print("MAE:", mae_train)
print("MSE:", mse_train)
print("RMSE:", rmse_train)
print("R2:", r2_train)

# 8️⃣ Test metrics
mae_test = mean_absolute_error(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)

print("\n=== TEST METRICS ===")
print("MAE:", mae_test)
print("MSE:", mse_test)
print("RMSE:", rmse_test)
print("R2:", r2_test)






Best parameters: {'model__max_depth': 15, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 5, 'model__n_estimators': 100}

=== TRAIN METRICS ===
MAE: 6.24653351712942
MSE: 63.11574949984498
RMSE: 7.94454212021341
R2: 0.8851699840377184

=== TEST METRICS ===
MAE: 10.799087154841391
MSE: 193.05911436306204
RMSE: 13.8945713990415
R2: 0.6136744584006477


In [131]:
import joblib

joblib.dump(grid, "engagement_model.pkl")

['engagement_model.pkl']

# Attrition model

In [132]:
# appearance_next_survey

In [133]:

x_columns=[  
    'department_HR', 'department_Operations','department_Finance', 'department_Sales', 'department_Marketing', 'department_Engineering', 
            'tenure_years', 'performance_rating', 'engagement_score',
       'feedback_category_Work-life balance','feedback_category_Career growth', 'feedback_category_Compensation','feedback_category_Culture', 'feedback_category_Management',
       'sentiment_Positive', 'sentiment_Negative', 'sentiment_Neutral',
       ]

        #  'lag_1_performance', 'lag_1_engagement'

y_column="appearance_next_survey"

df=df2.copy()

df = df.dropna(subset=["appearance_next_survey"])

X=df[x_columns]
y=df[y_column]

In [134]:

from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)


X_train.shape,X_test.shape


y_train = y_train.astype(int).astype(str)
y_test = y_test.astype(int).astype(str)

In [135]:
from sklearn.preprocessing import StandardScaler
# scaler=StandardScaler()

# X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])

# X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])


# X_train_sc=X_train

# X_test_sc=X_test



In [136]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd


# rfc = RandomForestClassifier(random_state=42)


pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(random_state=42))
])





param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}




grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=3,
    refit=True
)


grid.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__max_depth': [8, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__min_samples_leaf': [1, 2, ...], 'model__min_samples_split': [2, 5, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : 

In [137]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,f1_score,roc_auc_score


print("Best parameters:", grid.best_params_)


y_train_pred = grid.predict(X_train)
y_test_pred  = grid.predict(X_test)


print("--- Training Set ---")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("F1 Score:", f1_score(y_train, y_train_pred, average='weighted'))

# print("Classification Report:\n", classification_report(y_train, y_train_pred))
# print("Confusion Matrix:\n", confusion_matrix(y_train, y_train_pred))


print("--- Test Set ---")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("F1 Score:", f1_score(y_test, y_test_pred, average='weighted'))

# print("Classification Report:\n", classification_report(y_test, y_test_pred))
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))




Best parameters: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 4, 'model__min_samples_split': 2, 'model__n_estimators': 200}
--- Training Set ---
Accuracy: 0.8171091445427728
F1 Score: 0.8041329526562493
--- Test Set ---
Accuracy: 0.8470588235294118
F1 Score: 0.8406971183254923


In [138]:

import joblib

joblib.dump(grid, "Attrition_model.pkl")

['Attrition_model.pkl']

In [139]:
#### ml graph code 

In [140]:
from pathlib import Path
import sys

current_path = Path().resolve().parents[0] 

if str(current_path) not in sys.path:
    sys.path.append(str(current_path))




In [ ]:
# from llm_call import RoundRobinLLM
# from typing import Annotated, List, TypedDict
# from langchain_core.prompts import ChatPromptTemplate
# from pydantic import BaseModel
# from langchain_core.exceptions import OutputParserException
# from pydantic import BaseModel, Field
# from typing import Literal
# from langchain_core.output_parsers import PydanticOutputParser

# llm=RoundRobinLLM()


# class ml_instruct_class(BaseModel):
#     employee_id: str = Field(description="Employee ID extracted from user query")
    
#     prediction_model: Literal["Attrition_model","engagement_model","performance_model"] = Field(description="Type of prediction required")
    


# ml_instruct_prompt = ChatPromptTemplate.from_messages([
#     (
#         "system",
#         """
# You are an intelligent extraction engine.

# Your task:
# - Extract employee_id from the user query
# - Identify which prediction model is needed

# Model selection rules:
# - Attrition_model → if query mentions leaving, churn, retention, risk
# - engagement_model → if query mentions engagement, satisfaction
# - performance_model → if query mentions performance, rating, productivity

# Rules:
# - Return ONLY valid JSON
# - Do NOT explain anything
# - Do NOT hallucinate employee_id (return "" if not present)
# - Use exact model names only
# """
#     ),
#     (
#         "human",
#         """
# Previous conversation (use only if needed or related to current user query):
# {history}

# Current user query:
# {user_query}

# Output format:
# {{
#     "employee_id": "string",
#     "prediction_model": "Attrition_model | engagement_model | performance_model"
# }}
# """
#     )
# ])


# ml_instruct_chain = (
#     ml_instruct_prompt
#     | llm
#     | PydanticOutputParser(pydantic_object=ml_instruct_class)
# ).with_retry(
#     retry_if_exception_type=(OutputParserException,),
#     stop_after_attempt=3
# )






# encoder = joblib.load("onehot_encoder.pkl")

# # Load models
# models = {
#     "performance": joblib.load("performance_model.pkl"),
#     "attrition": joblib.load("attrition_model.pkl"),
#     "engagement": joblib.load("engagement_model.pkl")
# }

# def ml_node(user_query,history=None):
#     ml_instruct_result=ml_instruct_chain.invoke({
#         "history":history,
#         "user_query":user_query })
    

#     employee_id=ml_instruct_result.employee_id
#     prediction_model=ml_instruct_result.prediction_model


#     df=pd.read_csv(r"..\Synthetic_Data\feedback_sentiment_analysis.csv")
#     df["survey_date"] = pd.to_datetime(df["survey_date"], format="%d-%m-%Y")
    
#     filtered_df = df[df["employee_id"] == employee_id].sort_values(by="survey_date")
#     last_row_df=filtered_df.tail(1)
#     next_year = int(last_row_df["survey_date"].dt.year.iloc[0] + 1)
#     current_year = df["survey_date"].dt.year.max()

    
#     json_data = filtered_df.to_json(orient="records", date_format="iso")

#     for col in encoder.feature_names_in_:
#         if col not in last_row_df.columns:
#             last_row_df[col] = None

#     last_row_df = last_row_df[encoder.feature_names_in_]



#     print("====================================")
#     df_encoded = encoder.transform(last_row_df)

#     if prediction_model=="Attrition_model":
#         model=joblib.load("attrition_model.pkl")

#         # Ensure same columns as training
#         for col in model.feature_names_in_:
#             if col not in df_encoded.columns:
#                 df_encoded[col] = 0

#         df_encoded = df_encoded[model.feature_names_in_]

#         # Predict
#         prediction = model.predict_proba(df_encoded)[0][0]


        


#     elif prediction_model=="engagement_model":
#         model=joblib.load("engagement_model.pkl")

#         # Ensure same columns as training
#         for col in model.feature_names_in_:
#             if col not in df_encoded.columns:
#                 df_encoded[col] = 0

#         df_encoded = df_encoded[model.feature_names_in_]

#         # Predict
#         prediction = model.predict(df_encoded)[0]



#     elif prediction_model=="performance_model":
#         model=joblib.load("performance_model.pkl")

#         # Ensure same columns as training
#         for col in model.feature_names_in_:
#             if col not in df_encoded.columns:
#                 df_encoded[col] = 0

#         df_encoded = df_encoded[model.feature_names_in_]

#         # Predict
#         prediction = model.predict(df_encoded)[0]

#     prompt = f"""
# Previous conversation (only if relevant):
# {history}

# Current user query:
# {user_query}

# Employee data details:
# {json_data}

# Prediction type: {prediction_model}
# Prediction for employee in next year ({next_year}): {prediction}

# Consider the current year as {current_year} when interpreting this prediction.

# Generate a professional insight for this employee based on the above information.

# answer in 3 lines
# """
#     result=llm.invoke(prompt)

#     return result.content






In [ ]:
# user_query="attration of E001"

# pre=ml_node(user_query)

# print(pre)

C:\Users\shubham.sharma\AppData\Local\Temp\ipykernel_7056\1494731164.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_row_df[col] = None
C:\Users\shubham.sharma\AppData\Local\Temp\ipykernel_7056\1494731164.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_row_df[col] = None
C:\Users\shubham.sharma\AppData\Local\Temp\ipykernel_7056\1494731164.py:107: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value i

- E001 shows a modest attrition risk (~21% probability), indicating a generally stable but not negligible chance of departure.  
- Strong performance (rating 4.0) and positive feedback on flexible hours suggest high satisfaction with work‑life balance.  
- To further reduce turnover risk, continue offering schedule flexibility, provide targeted growth opportunities, and regularly check engagement levels.


In [ ]:
# print(pre)

E001 shows a moderate attrition risk (~21% probability), indicating they are not imminently likely to leave but warrant attention. Their strong performance (rating 4.0), positive sentiment, and appreciation for flexible hours suggest high engagement in work‑life balance. Continue offering flexible scheduling, provide targeted development opportunities, and schedule regular check‑ins to reinforce retention.


,employee_id,survey_date,department,tenure_years,performance_rating,engagement_score,feedback_comment,feedback_category,sentiment,summary
0,E001,2019-01-01,Operations,0.9,4.0,25,The flexible hours have really helped me maint...,Work-life balance,Positive,Flexible hours improve work-life balance
